# **PVSystem** class workflow: Creating PV systems from high-level specifications

The `PVSystem` class helps a user create and run energy calculations with high-level specifications of a PV plant. This workflow is designed to be flexible to match the degree of design detail available to users for a given project. When some specifications of a PV plant is not provided but required, `PVSystem` class will assume default effects and losses where appropriate.

This notebook shows how to define the specifications for a project and how to run an energy calculation using the SolarFarmer Python SDK.

See details related to the PVSystem class in SolarFarmer's public documentation:
- [Acquiring API key](https://mysoftware.dnv.com/download/public/renewables/solarfarmer/manuals/latest/WebApi/Introduction/ApiKey.html)
- [ModelChain endpoint](https://mysoftware.dnv.com/download/public/renewables/solarfarmer/manuals/latest/WebApi/Endpoints/ModelChainEndpoint.html)
- [API output results](https://mysoftware.dnv.com/download/public/renewables/solarfarmer/manuals/latest/WebApi/Endpoints/ModelChainEndpoint.html#outputs)

> **Note:** `PVSystem` generates an approximated plant design by inferring layout geometry, string sizing, and losses from high-level inputs. Results are intended for early-stage yield screening and comparative analysis. For full design fidelity, export a detailed payload from SolarFarmer Desktop or use Workflow 3.

## 0. Prerequisites

**Notebook Information:**
- **Last Updated:** March 2026
- **Written for:** SolarFarmer SDK v0.1.0+
- [View latest version in repository](https://...)

### 0.1 Install the SolarFarmer Python SDK

This notebook requires the SolarFarmer Python SDK to be installed. Install it via pip:

```bash
pip install solarfarmer
```

In [1]:
import os
import solarfarmer as sf
from pathlib import Path

# Check SDK version compatibility
NOTEBOOK_MIN_SDK_VERSION = "0.1.0"

print(f"SolarFarmer Python SDK v{sf.__version__}")

# Parse versions for comparison
def parse_version(v):
    """Simple version parser for x.y.z format"""
    return tuple(map(int, v.split('.')))

try:
    if parse_version(sf.__version__) < parse_version(NOTEBOOK_MIN_SDK_VERSION):
        print(f"\n    WARNING: This notebook requires SDK v{NOTEBOOK_MIN_SDK_VERSION} or later.")
        print(f"    Your version: {sf.__version__}")
        print(f"    Some examples may not work correctly.")
        print(f"    Upgrade with: pip install --upgrade solarfarmer\n")
except Exception:
    # If version parsing fails, just continue
    pass

sf.configure_logging();


SolarFarmer Python SDK v0.1.0


### 0.2 API Key Required

You need a SolarFarmer API key to run the calculations in this notebook. Instructions for acquiring one is [HERE](https://mysoftware.dnv.com/download/public/renewables/solarfarmer/manuals/latest/WebApi/Introduction/ApiKey.html)

**Important:** Avoid hardcoding your API key directly in notebook cells.

- **Use environment variables (Recommended):**

  The SDK automatically uses the `SF_API_KEY` environment variable. Set it in your terminal before starting Jupyter:

  **Linux/Mac:**
  ```bash
  export SF_API_KEY="your-key-here"
  ```

  **Windows:**
  ```bash
  set SF_API_KEY=your-key-here
  ```

- **Entering your API key (Alternative):**

  This notebook will prompt you to enter your API key and keep it hidden from view.

In [2]:
if os.getenv("SF_API_KEY") is None:
    print("WARNING: `SF_API_KEY` environment variable not set.")
    import getpass
    api_key = getpass.getpass("Enter your SolarFarmer API key: ")
    print("Using API key entered by user.\n")
else:
    api_key = os.getenv("SF_API_KEY")
    print("Using API key from environment variable `SF_API_KEY`\n")

Using API key from environment variable `SF_API_KEY`



### 0.3 Sample Data Required

This notebook uses sample input files (meteorological data, PAN and OND files).

**Download the sample data:**

Clone the full repository:
```bash
git clone tbd.git
```

Or download the `docs/notebooks/sample_data/` folder directly from the repository:

tbd url

The examples below assume the data is in a `sample_data/` folder relative to this notebook, but you can place it anywhere and update the path in the next cell.

In [3]:
# Path to the sample_data folder containing input files
# Modify this path if you have placed that data elsewhere
sample_data_dir = Path("sample_data")

# Verify the path exists
if not sample_data_dir.exists():
    print(f"WARNING: Sample data not found at: {sample_data_dir.absolute()}")
    print("Please download from: https://...")  #TODO: Add actual URL
    print("Or update the 'sample_data_dir' path above.")
else:
    print(f"Sample data found at: {sample_data_dir.absolute()}")

Sample data found at: /home/iantse/git_repos/solarfarmer-api-python-sdk/docs/notebooks/sample_data


## 1. Defining high-level project specifications using the `PVSystem` class

The `PVSystem` class contains several groups of specifications including location, layout and mounting details, site capacity and grid constraints, major equipment (inverter and module) used, and a number of specific modelchain losses. 

The configuration of specs with the `PVSystem` class is introduced by topics in this example notebook. First, an example using the minimal recommended fields. Then, a second one with further settings that override the defaults.

### 1.1 Example with minimal recommended specifications

In [4]:
# Initiate the PVSystem class
simple_plant = sf.PVSystem()
simple_plant.name = "Example Plant using high-level configuration"

# Location details
simple_plant.latitude = 46.95
simple_plant.longitude = 7.37
simple_plant.altitude = 575.0
simple_plant.timezone = "Europe/Amsterdam"

# Capacity and layout details
simple_plant.dc_capacity_MW = 12.0
simple_plant.ac_capacity_MW = 10.0
simple_plant.grid_limit_MW = 9.0
simple_plant.mounting = "Fixed"  # Options: 'Fixed' or 'Tracker'
simple_plant.gcr = 0.4
simple_plant.tilt = 45
simple_plant.azimuth = 225 # Southwest facing
simple_plant.flush_mount = False
simple_plant.bifacial = True
simple_plant.inverter_type = "Central"  # Options: 'String' or 'Central'
simple_plant.transformer_stages = 1 # Options: 0 (ideal/no-loss transformer) or 1 (with transformer)

# Auxiliary files
simple_plant.pan_files = {"Module1": sample_data_dir / "Inputs_Bern_2D_racks" / "CanadianSolar_CS6U-330M_APP.PAN"}
simple_plant.ond_files = {"Inverter1": sample_data_dir / "Inputs_Bern_2D_racks" / "Sungrow_SG125HV_APP.OND"}
simple_plant.weather_file = sample_data_dir / "Inputs_Bern_2D_racks" / "Bern-hour.dat"

# Desired energy calculation timeseries outputs
simple_plant.generate_loss_tree_timeseries = True
simple_plant.generate_pvsyst_format_timeseries = True

#### 1.1.1 Viewing the PVSystem configuration

Use the method `describe()` to get the info contained in the PVSystem object after being set up or review the default values. The variable `verbose` controls the amount of information printed out.

In [5]:
simple_plant.describe(verbose=True)


PV PLANT SUMMARY

--- BASIC METADATA ---
Name: Example Plant using high-level configuration
Results Available: False

--- LOCATION ---
Latitude: 46.95°
Longitude: 7.37°
Altitude: 575.0 m
Timezone: Europe/Amsterdam

--- CAPACITY ---
DC Capacity: 12.0 MW
AC Capacity: 10.0 MW
Grid Limit: 9.0 MW

--- ARRAY CONFIGURATION ---
GCR (Ground Coverage Ratio): 0.4
Pitch: None m
Tilt: 45°
Azimuth: 225°

--- MOUNTING & ORIENTATION ---
Mounting Type: Fixed
Module Orientation: OrientationType.PORTRAIT
Modules Across: 1
Mounting Height: 0.7 m
Flush Mount: False

--- BIFACIAL PARAMETERS ---
Bifacial Modules: True
  - Transmission Factor: 0.0
  - Shade Loss Factor: 0.0
  - Mismatch Loss Factor: 0.0

--- INVERTER & TRANSFORMER ---
Inverter Type: Central
String Length: None modules
Transformer Stages: 1 (0=Ideal/NoLoss, 1=MV/HV transformer)

--- LOSSES ---
DC Ohmic Loss: 0.015 (per unit)
AC Ohmic Loss: 0.005 (per unit)
Module Mismatch Loss: 0.005 (per unit)
Module Quality Factor: 0.0 (per unit)
LID Loss: 

#### 1.1.2 Running the energy calculation

Use the method `run_energy_calculation` to perform the energy calculation via the API.

In [6]:
simple_plant.run_energy_calculation(
    project_id="my_project_id", 
    api_version="v6",
    api_key=api_key,
    save_outputs=False,
)

INFO: Making API call to ModelChain (2D) endpoint: https://solarfarmer.dnv.com/v6/api/ModelChain
Start time = 05-03-2026 14:18:13



DESIGN SUMMARY

Capacity Summary:
  Target (input) DC Capacity: 12.0 MW
  Design (output for simulation) DC Capacity: 11.991 MW
  Target (input) AC Capacity: 10.0 MW
  Design (output for simulation) AC Capacity: 10.000 MW

Layout Configuration:
  Number of Inverters: 80
  Number of Modules: 36363
  String Length: 29
  Total Strings: 2320



INFO: SUCCESS: Calculation returned successfully (time taken: 6.6 seconds)
/home/iantse/git_repos/solarfarmer-api-python-sdk/solarfarmer/models/energy_calculation_results.py:1771: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data["date"] = pd.to_datetime(data["date"], utc=True).dt.tz_localize(None)


-------------------------------------------------------
		General site data:
-------------------------------------------------------
Project name = my_project_id
Location (latitude, longitude, altitude) = 46.95000°, 7.37000°, 575.0 m
2D/3D = 2D calculation
Mounting type = Fixed-tilt racks
AC capacity of system = 10 MW
DC capacity of system = 11.99121 MW
Run in SolarFarmer API version 6.0.21
-------------------------------------------------------
    Annual Performance Summary (project year 1, 1990)
-------------------------------------  -----------------
Average annual temperature             10.61°C
Horizontal irradiation                 1192.73 kWh/m²
Irradiation on tilted plane            1282.73 kWh/m²
Effective irradiation on tilted plane  1164.71 kWh/m²
Energy yield                           959.58 kWh/kWp
Net energy                             11506.52 MWh/year
Performance Ratio                      0.7481
-------------------------------------  -----------------


#### 1.1.3 Accessing the results

Use the variable `results` to access the simulation results in a `CalculationResults` class

In [7]:
simple_plant.results.print_annual_results()


----------------------------------------
Annual Results Summary (Years: 1 (1990))
----------------------------------------
Property                     Units            Year 1
                                              (1990)
---------------------------  --------  -------------
Energy yield                 kWh/kWp          959.58
Net energy                   MWh/year      11,506.52
Production DC                kWh       12,025,493.79
Production AC                kWh       11,768,464.63
Performance Ratio            %                 74.81
Performance Ratio Bifacial   %                 74.81
Average annual temperature   °C                10.61
Horizontal irradiation       kWh/m²         1,192.73
Irradiation on tilted plane  kWh/m²         1,282.73
GI with horizon              kWh/m²         1,282.73
Global effective irradiance  kWh/m²         1,164.71
Gain on tilted plane         %                  7.55
\n----------------------------------------
Annual Effects Summary (Years: 1 (1990

### 1.2 Example with further effects and settings

When you have further information about the PV plant, you can go deeper in the definition of effects (gains and losses).

In [8]:
advanced_plant = simple_plant.make_copy() # Creates a copy of the plant object without results.

# Horizon information
advanced_plant.horizon(elevation_angles=[0, 5, 10, 20], azimuth_angles=[0, 90, 180, 270])

# Update soiling data
advanced_plant.soiling_loss = [0.14] * 12  # Override via property (e.g., 0.14 for 14% loss)

# Update albedo data
advanced_plant.albedo = [0.25, 0.3] * 6 # monthly values (e.g., 0.25 for 25% albedo)

# Override several default module-level properties (if desired)
advanced_plant.module_mismatch = 0.005 # per unit (e.g., 0.005 for 0.5%)
advanced_plant.module_quality_factor = 0.01 # per unit (it can be positive or negative, e.g., 0.01 for 1% gain or -0.01 for 1% loss)

# Auxiliary losses - simple fractional loss as a fraction of AC output
advanced_plant.aux_loss_fixed_factor = 0.005  # 0.5% of AC output
advanced_plant.aux_loss_apply_at_night = True  # Also apply loss at night


#### 1.2.1 Running the energy calculation with detailed settings

Run the energy calculation of the plant with more detailed settings

In [9]:
advanced_plant.run_energy_calculation(
    project_id="my_project_id",
    api_version="v6",
    api_key=api_key,
    save_outputs=False,
)

INFO: Making API call to ModelChain (2D) endpoint: https://solarfarmer.dnv.com/v6/api/ModelChain
Start time = 05-03-2026 14:18:20



DESIGN SUMMARY

Capacity Summary:
  Target (input) DC Capacity: 11.99121 MW
  Design (output for simulation) DC Capacity: 11.991 MW
  Target (input) AC Capacity: 10.0 MW
  Design (output for simulation) AC Capacity: 10.000 MW

Layout Configuration:
  Number of Inverters: 80
  Number of Modules: 36337
  String Length: 29
  Total Strings: 2320



INFO: SUCCESS: Calculation returned successfully (time taken: 6.1 seconds)
/home/iantse/git_repos/solarfarmer-api-python-sdk/solarfarmer/models/energy_calculation_results.py:1771: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data["date"] = pd.to_datetime(data["date"], utc=True).dt.tz_localize(None)


-------------------------------------------------------
		General site data:
-------------------------------------------------------
Project name = my_project_id
Location (latitude, longitude, altitude) = 46.95000°, 7.37000°, 575.0 m
2D/3D = 2D calculation
Mounting type = Fixed-tilt racks
AC capacity of system = 10 MW
DC capacity of system = 11.99121 MW
Run in SolarFarmer API version 6.0.21
-------------------------------------------------------
    Annual Performance Summary (project year 1, 1990)
-------------------------------------  ----------------
Average annual temperature             10.61°C
Horizontal irradiation                 1192.73 kWh/m²
Irradiation on tilted plane            1295.80 kWh/m²
Effective irradiation on tilted plane  914.09 kWh/m²
Energy yield                           788.88 kWh/kWp
Net energy                             9459.62 MWh/year
Performance Ratio                      0.6088
-------------------------------------  ----------------


In [10]:
advanced_plant.results.print_annual_results()


----------------------------------------
Annual Results Summary (Years: 1 (1990))
----------------------------------------
Property                     Units           Year 1
                                             (1990)
---------------------------  --------  ------------
Energy yield                 kWh/kWp         788.88
Net energy                   MWh/year      9,459.62
Production DC                kWh       9,887,837.17
Production AC                kWh       9,670,401.01
Performance Ratio            %                60.88
Performance Ratio Bifacial   %                60.88
Average annual temperature   °C               10.61
Horizontal irradiation       kWh/m²        1,192.73
Irradiation on tilted plane  kWh/m²        1,295.80
GI with horizon              kWh/m²        1,159.80
Global effective irradiance  kWh/m²          914.09
Gain on tilted plane         %                 8.64
\n----------------------------------------
Annual Effects Summary (Years: 1 (1990))
------------

## 2. Accessing the JSON API without running the energy calculation

Before running the energy calculation via SolarFarmer's API, you can build the JSON API to visualize it, and also save it to a file via the methods:
- `produce_payload()` produces the JSON API, this is available in the variable `payload`.
- `payload_to_file()` produces the JSON API, saves it to an external file path.

In [11]:
simple_plant.produce_payload()
simple_plant.payload


DESIGN SUMMARY

Capacity Summary:
  Target (input) DC Capacity: 11.99121 MW
  Design (output for simulation) DC Capacity: 11.991 MW
  Target (input) AC Capacity: 10.0 MW
  Design (output for simulation) AC Capacity: 10.000 MW

Layout Configuration:
  Number of Inverters: 80
  Number of Modules: 36337
  String Length: 29
  Total Strings: 2320



'{"location":{"longitude":7.37,"latitude":46.95,"altitude":575.0},"pvPlant":{"transformers":[{"transformerCount":1,"inverters":[{"inverterSpecID":"Sungrow_SG125HV_APP","inverterCount":53,"layouts":[{"layoutCount":1,"moduleSpecificationID":"CanadianSolar_CS6U-330M_APP","mountingTypeID":"Fixed-Tilt Rack Template 1","isTrackers":false,"azimuth":225.0,"pitch":4.8999999999999995,"totalNumberOfStrings":16,"stringLength":29,"inverterInput":[0],"dcOhmicConnectorLoss":0.015,"moduleMismatchLoss":0.005,"name":"layout_1","numberOfStringsInFrontRow":0,"numberOfStringsInBackRow":0,"numberOfStringsInRightRow":0,"numberOfStringsInLeftRow":0,"numberOfStringsInIsolatedRow":0,"terrainAzimuth":0.0,"terrainSlope":0.0,"moduleQualityFactor":0.0}],"acWiringOhmicLoss":0.005,"name":"Inverter_1"},{"inverterSpecID":"Sungrow_SG125HV_APP","inverterCount":27,"layouts":[{"layoutCount":1,"moduleSpecificationID":"CanadianSolar_CS6U-330M_APP","mountingTypeID":"Fixed-Tilt Rack Template 1","isTrackers":false,"azimuth":225

In [12]:
simple_plant.payload_to_file(sample_data_dir / "PVSystem_modelchain_payload.json")